# S22 — ViT & Multimodal

**Week 12 · Mon Nov 9, 2026 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s22_vit_multimodal.ipynb)

Every cell below is a worked example from the [S22 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s22/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s22.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s22.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## An image is a sequence of patches


*Expected output starts with:* `images  : (2, 3, 32, 32)`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

B, C, H, W = 2, 3, 32, 32       # batch of 2 RGB images, 32x32
P = 4                            # patch size
D = 96                           # embedding dimension
n_patches = (H // P) * (W // P)  # 8 * 8 = 64 patches per image

imgs = torch.randn(B, C, H, W)

def patchify(x, p):
    """(B, C, H, W) -> (B, n_patches, p*p*C), row-major patch order."""
    B, C, H, W = x.shape
    x = x.reshape(B, C, H // p, p, W // p, p)   # split H and W into blocks
    x = x.permute(0, 2, 4, 1, 3, 5)             # (B, H/p, W/p, C, p, p)
    return x.reshape(B, (H // p) * (W // p), C * p * p)

def unpatchify(patches, p, c, h, w):
    """Inverse of patchify."""
    B = patches.shape[0]
    x = patches.reshape(B, h // p, w // p, c, p, p)
    x = x.permute(0, 3, 1, 4, 2, 5)
    return x.reshape(B, c, h, w)

patches = patchify(imgs, P)
print(f"images  : {tuple(imgs.shape)}")
print(f"patches : {tuple(patches.shape)}   (expect ({B}, {n_patches}, {C * P * P}))")

# round-trip check: patchify must lose no information
recon = unpatchify(patches, P, C, H, W)
print(f"round-trip max |x - unpatchify(patchify(x))| = {(imgs - recon).abs().max().item():.1f}")

# linear projection to embeddings + [CLS] token + position embeddings
proj = nn.Linear(C * P * P, D)
cls_token = nn.Parameter(torch.zeros(1, 1, D))
pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, D) * 0.02)

tokens = proj(patches)                              # (B, 64, 96)
tokens = torch.cat([cls_token.expand(B, -1, -1), tokens], dim=1)
tokens = tokens + pos_embed
print(f"tokens  : {tuple(tokens.shape)}   (expect ({B}, {n_patches + 1}, {D}))")

# the same projection as a strided convolution (the trick real ViTs use)
conv = nn.Conv2d(C, D, kernel_size=P, stride=P)
with torch.no_grad():
    conv.weight.copy_(proj.weight.reshape(D, C, P, P))
    conv.bias.copy_(proj.bias)
conv_tokens = conv(imgs).flatten(2).transpose(1, 2)  # (B, 64, 96)
diff = (conv_tokens - proj(patches)).abs().max()
print(f"max |Linear-on-patches - strided Conv2d| = {diff.item():.2e}")

## The bias experiment on your CPU


*Expected output starts with:* `train size     model   params  test acc`


In [ ]:
# Inductive bias vs data scale, in miniature: a small CNN and a small ViT
# on the same local-pattern task, same optimizer, same step budget, at two
# training-set sizes. The task: is the 3x3 shape hidden in the noise a
# "plus" or an "X"? Locality and translation equivariance should help.
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

PLUS = torch.tensor([[0., 1, 0], [1, 1, 1], [0, 1, 0]]) * 2.0
X    = torch.tensor([[1., 0, 1], [0, 1, 0], [1, 0, 1]]) * 2.0

def make_data(n, gen):
    imgs = 0.3 * torch.randn(n, 1, 16, 16, generator=gen)
    labels = torch.randint(0, 2, (n,), generator=gen)
    pos = torch.randint(0, 13, (n, 2), generator=gen)   # top-left of the 3x3 patch
    for i in range(n):
        r, c = pos[i]
        imgs[i, 0, r:r + 3, c:c + 3] += PLUS if labels[i] == 0 else X
    return imgs, labels

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))

    def forward(self, x):
        return self.net(x)

class SmallViT(nn.Module):
    def __init__(self, d=48, n_head=4, n_layer=2, p=4):
        super().__init__()
        self.embed = nn.Conv2d(1, d, kernel_size=p, stride=p)   # patchify
        self.pos = nn.Parameter(torch.randn(1, (16 // p) ** 2, d) * 0.02)
        enc = nn.TransformerEncoderLayer(d, n_head, 4 * d, batch_first=True,
                                         dropout=0.0, norm_first=True)
        self.blocks = nn.TransformerEncoder(enc, n_layer, enable_nested_tensor=False)
        self.head = nn.Linear(d, 2)

    def forward(self, x):
        tokens = self.embed(x).flatten(2).transpose(1, 2) + self.pos
        return self.head(self.blocks(tokens).mean(dim=1))       # mean-pool

def train_eval(model, Xtr, ytr, Xte, yte, steps=400):
    gen = torch.Generator().manual_seed(2)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    for step in range(steps):
        ix = torch.randint(0, len(Xtr), (64,), generator=gen)
        loss = F.cross_entropy(model(Xtr[ix]), ytr[ix])
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

gen = torch.Generator().manual_seed(1)
Xte, yte = make_data(1000, gen)

print(f"{'train size':>10} {'model':>9} {'params':>8} {'test acc':>9}")
for n_train in [256, 4096]:
    Xtr, ytr = make_data(n_train, gen)
    for name, cls in [("CNN", SmallCNN), ("ViT", SmallViT)]:
        torch.manual_seed(0)                 # identical init treatment
        model = cls()
        n_params = sum(p.numel() for p in model.parameters())
        acc = train_eval(model, Xtr, ytr, Xte, yte)
        print(f"{n_train:>10} {name:>9} {n_params:>8,} {acc:>9.4f}")

## Reading a ViT's attention maps


*Expected output starts with:* `test accuracy: 0.9880`


In [ ]:
# Where does a trained ViT look? Train a tiny ViT with a [CLS] token on the
# plus-vs-X task, then read out the last block's CLS->patch attention weights
# and check whether they land on the patches that contain the shape.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

PLUS = torch.tensor([[0., 1, 0], [1, 1, 1], [0, 1, 0]]) * 2.0
X    = torch.tensor([[1., 0, 1], [0, 1, 0], [1, 0, 1]]) * 2.0

def make_data(n, gen):
    imgs = 0.3 * torch.randn(n, 1, 16, 16, generator=gen)
    labels = torch.randint(0, 2, (n,), generator=gen)
    pos = torch.randint(0, 13, (n, 2), generator=gen)
    for i in range(n):
        r, c = pos[i]
        imgs[i, 0, r:r + 3, c:c + 3] += PLUS if labels[i] == 0 else X
    return imgs, labels, pos

class Block(nn.Module):
    """Pre-norm encoder block with attention weights exposed."""
    def __init__(self, d, n_head):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n_head, self.d = n_head, d

    def forward(self, x):
        B, T, d = x.shape
        h = self.ln1(x)
        q, k, v = self.qkv(h).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, self.n_head, -1).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(d // self.n_head)
        att = att.softmax(-1)
        self.last_att = att.detach()                 # (B, heads, T, T)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, d))
        return x + self.mlp(self.ln2(x))

class ViT(nn.Module):
    def __init__(self, d=48, n_head=4, n_layer=2, p=4):
        super().__init__()
        self.embed = nn.Conv2d(1, d, kernel_size=p, stride=p)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, 17, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, n_head) for _ in range(n_layer)])
        self.head = nn.Linear(d, 2)

    def forward(self, x):
        t = self.embed(x).flatten(2).transpose(1, 2)          # (B, 16, d)
        t = torch.cat([self.cls.expand(len(x), -1, -1), t], 1) + self.pos
        for blk in self.blocks:
            t = blk(t)
        return self.head(t[:, 0])                             # classify from CLS

gen = torch.Generator().manual_seed(1)
Xtr, ytr, _ = make_data(4096, gen)
Xte, yte, pos_te = make_data(500, gen)

model = ViT()
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
sample_gen = torch.Generator().manual_seed(2)
for step in range(400):
    ix = torch.randint(0, len(Xtr), (64,), generator=sample_gen)
    loss = F.cross_entropy(model(Xtr[ix]), ytr[ix])
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()

with torch.no_grad():
    acc = (model(Xte).argmax(1) == yte).float().mean()
    # CLS row of the last block's attention, averaged over heads: (B, 17)
    cls_att = model.blocks[-1].last_att.mean(dim=1)[:, 0, :]
    patch_att = cls_att[:, 1:]                                # drop CLS->CLS
print(f"test accuracy: {acc.item():.4f}")

# does the most-attended patch overlap the 3x3 shape?
hits = 0
for i in range(len(Xte)):
    r, c = pos_te[i]
    overlapping = {(rr // 4) * 4 + (cc // 4)
                   for rr in range(r, r + 3) for cc in range(c, c + 3)}
    hits += int(patch_att[i].argmax().item() in overlapping)
print(f"most-attended patch overlaps the shape: {hits}/{len(Xte)} "
      f"({100 * hits / len(Xte):.1f}%)  [chance would be ~13%]")

# one image in detail: attention over the 4x4 patch grid
i = 0
r, c = pos_te[i]
print(f"\nimage 0: shape top-left at pixel ({r},{c}) "
      f"-> patch grid rows {r // 4}-{(r + 2) // 4}, cols {c // 4}-{(c + 2) // 4}")
print("CLS attention over patches (x100, row-major 4x4 grid):")
grid = patch_att[i].reshape(4, 4) * 100
for row in grid:
    print("  " + " ".join(f"{v:5.1f}" for v in row.tolist()))

## CLIP: two encoders, one space


*Expected output starts with:* `chance-level loss for batch of 64: ln(64) = 4.1589`


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

N, D_IMG, D_TXT, D = 64, 48, 32, 16   # 64 pairs, different raw dims, shared space

# Synthetic "images" and "captions": pair i shares a hidden concept vector z_i,
# seen through two different random linear "modalities" plus noise.
g = torch.Generator().manual_seed(1)
z = torch.randn(N, 8, generator=g)
img_map = torch.randn(8, D_IMG, generator=g)
txt_map = torch.randn(8, D_TXT, generator=g)
imgs = z @ img_map + 0.1 * torch.randn(N, D_IMG, generator=g)
txts = z @ txt_map + 0.1 * torch.randn(N, D_TXT, generator=g)

img_proj = nn.Linear(D_IMG, D)
txt_proj = nn.Linear(D_TXT, D)
log_temp = nn.Parameter(torch.tensor(0.0))   # learnable temperature, like CLIP

def clip_loss():
    ie = F.normalize(img_proj(imgs), dim=-1)      # unit-norm embeddings
    te = F.normalize(txt_proj(txts), dim=-1)
    logits = ie @ te.T * log_temp.exp()           # (N, N) similarity matrix
    labels = torch.arange(N)                      # pair i matches pair i
    loss_i = F.cross_entropy(logits, labels)      # image -> which caption?
    loss_t = F.cross_entropy(logits.T, labels)    # caption -> which image?
    return (loss_i + loss_t) / 2, ie, te

params = list(img_proj.parameters()) + list(txt_proj.parameters()) + [log_temp]
opt = torch.optim.Adam(params, lr=1e-2)

print(f"chance-level loss for batch of {N}: ln({N}) = {math.log(N):.4f}")
for step in range(301):
    loss, ie, te = clip_loss()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 75 == 0:
        with torch.no_grad():
            sims = ie @ te.T
            matched = sims.diag().mean()
            mismatched = (sims.sum() - sims.diag().sum()) / (N * N - N)
            top1 = (sims.argmax(dim=1) == torch.arange(N)).float().mean()
        print(f"step {step:3d}  loss {loss.item():.4f}  "
              f"matched sim {matched.item():+.4f}  "
              f"mismatched sim {mismatched.item():+.4f}  "
              f"retrieval@1 {top1.item():.4f}")

## Temperature, and a sigmoid alternative


*Expected output starts with:* `part 1 - temperature (softmax InfoNCE, batch 64):`


In [ ]:
# Two questions about the contrastive recipe, answered by experiment:
# (1) what does the temperature do?  (2) how does SigLIP's pairwise sigmoid
# loss compare to softmax InfoNCE when the training batch is small?
# Retrieval is always evaluated against a 256-pair held-out pool.
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

D_IMG, D_TXT, D = 48, 32, 16

def make_pairs(n, gen, img_map, txt_map):
    z = torch.randn(n, 8, generator=gen)
    imgs = z @ img_map + 2.0 * torch.randn(n, D_IMG, generator=gen)
    txts = z @ txt_map + 2.0 * torch.randn(n, D_TXT, generator=gen)
    return imgs, txts

gen = torch.Generator().manual_seed(1)
img_map = torch.randn(8, D_IMG, generator=gen)
txt_map = torch.randn(8, D_TXT, generator=gen)
tr_imgs, tr_txts = make_pairs(512, gen, img_map, txt_map)
te_imgs, te_txts = make_pairs(256, gen, img_map, txt_map)

def run(loss_type, batch_size, temp_mode="learnable", steps=400):
    torch.manual_seed(0)
    img_proj, txt_proj = nn.Linear(D_IMG, D), nn.Linear(D_TXT, D)
    log_t = nn.Parameter(torch.tensor(0.0))          # temperature t = exp(log_t)
    bias = nn.Parameter(torch.tensor(-10.0))         # SigLIP's learnable bias
    params = list(img_proj.parameters()) + list(txt_proj.parameters()) + [bias]
    if temp_mode == "learnable":
        params.append(log_t)
    else:                                            # fixed temperature
        with torch.no_grad():
            log_t.fill_(torch.tensor(1.0 / temp_mode).log().item())
    opt = torch.optim.Adam(params, lr=1e-2)
    bgen = torch.Generator().manual_seed(2)
    for step in range(steps):
        ix = torch.randint(0, 512, (batch_size,), generator=bgen)
        ie = F.normalize(img_proj(tr_imgs[ix]), dim=-1)
        te = F.normalize(txt_proj(tr_txts[ix]), dim=-1)
        sims = ie @ te.T * log_t.exp()
        if loss_type == "softmax":                   # symmetric InfoNCE
            labels = torch.arange(batch_size)
            loss = (F.cross_entropy(sims, labels)
                    + F.cross_entropy(sims.T, labels)) / 2
        else:                                        # SigLIP: pairwise sigmoid
            signs = 2 * torch.eye(batch_size) - 1    # +1 diagonal, -1 elsewhere
            loss = -F.logsigmoid(signs * (sims + bias)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    final_loss = loss.item()
    with torch.no_grad():                            # retrieval over 256-pool
        ie = F.normalize(img_proj(te_imgs), dim=-1)
        te = F.normalize(txt_proj(te_txts), dim=-1)
        top1 = ((ie @ te.T).argmax(1) == torch.arange(256)).float().mean()
    return top1.item(), log_t.exp().item(), final_loss

print("part 1 - temperature (softmax InfoNCE, batch 64):")
print(f"{'temperature':>20} {'retrieval@1':>12} {'final T':>9} {'train loss':>11}")
for mode, label in [(1.0, "fixed T=1.0"), (0.07, "fixed T=0.07"),
                    ("learnable", "learnable")]:
    top1, t, fl = run("softmax", 64, mode)
    print(f"{label:>20} {top1:>12.4f} {1 / t:>9.4f} {fl:>11.4f}")

print("\npart 2 - loss type x batch size (learnable temperature):")
print(f"{'loss':>9} {'batch':>6} {'retrieval@1':>12}")
for loss_type in ["softmax", "sigmoid"]:
    for bs in [8, 64]:
        top1, _, _ = run(loss_type, bs)
        print(f"{loss_type:>9} {bs:>6} {top1:>12.4f}")

## Try it yourself

1. Train a small classifier on the toy data twice: once on raw pixels of `unpatchify`-scrambled images (shuffle the patch order with a fixed permutation) and once on originals, using the same MLP. Position information is destroyed by the shuffle for the MLP either way — explain why the accuracies do (or do not) differ.
2. Extend the patchify script to `P = 8` and `P = 16` on a 32x32 image, printing sequence length and per-patch dimension for each. What is the trade-off as `P` grows, and what happens to attention cost as `P` shrinks?
3. In the CLIP script, reduce the shared-concept dimension from 8 to 1 and rerun. What happens to retrieval@1 and to the matched-pair similarity, and why?
4. Break the pairing deliberately: shuffle `txts` so pair `i`'s caption belongs to a different image, and retrain. Where does the loss plateau relative to `ln(64)`, and what does that tell you about what the loss measures?


---

Full discussion of everything above: [S22 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s22/).
